# Question transformations

# Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import getpass

/Users/basharnaji/Documents/GitHub/building-llm-applications/ch09/env_ch09/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

uk_granular_collection.reset_collection() #A

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [3]:
from langchain_text_splitters import HTMLSectionSplitter
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [5]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

In [6]:
headers_to_split_on = [("h1", "Header 1"),("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [7]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #B
        temp_chunks = html_section_splitter.split_text(
            html_string) #C
        h2_temp_chunks = [chunk for chunk in 
                          temp_chunks if "Header 2" 
                          in chunk.metadata] #D
        all_chunks.extend(h2_temp_chunks) 

    return all_chunks

In [8]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url) #E
    docs =  html_loader.load() #F
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(
            documents=granular_chunks)

#A In case it exists
#B Extract the HTML text from the document
#C Each chunk is a H1 or H2 HTML section
#D Only keep content associated with H2 sections        
#E Loader for one destination
#F Documents of one destination

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.24it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.39it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.38it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.59it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.15it/s]

{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


# Rewrite-retrieve-read

## Retrieving content with original user question

In [9]:
user_question = "Tell me some fun things I can enjoy in Cornwall"
initial_results = uk_granular_collection.similarity_search(
    query=user_question,k=4)
for doc in initial_results:
    print(doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 
 
 4.1   By train 
 
 
 
 
 
 
 4.2   By car 
 
 
 
 
 
 
 4.3   By plane 
 
 
 
 
 
 
 
 
 5   Get around 
 
 
 
 
 5.1   By bus 
 
 
 
 
 
 
 5.2   By train 
 
 
 
 
 
 
 
 
 6   See 
 
 
 
 
 6.1   National Trust properties 
 
 
 
 
 
 
 
 
 7   Do 
 
 
 
 
 7.1   Festivals 
 
 
 
 
 
 
 
 
 8   Drink 
 
 
 
 
 
 
 9   Stay safe 
 
 
 
 
 
 
 10   Go next 
 
 
 
 
 
 
 
 
 
 
 
 
 
 North Cornwall  is in  Cornwall . It includes much of the Cornish coast along the Celtic Sea and some top surfing areas.' metadata={'Header 2': 'Contents'}
page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 
 
 4.1   By train 
 
 
 
 
 
 
 4.2   By car 
 
 
 
 
 
 
 
 
 5   Get around 
 
 
 
 
 5.1   By bus 
 
 
 
 
 
 
 5.2   By train 
 
 


In [10]:
# COMMENT: the retrieval from the vector store against the original question is bad

## Question rewrite

### Setting up the query rewriter chain

In [10]:
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [11]:
ANTHROPIC_API_KEY = getpass.getpass('Enter your ANTHROPIC_API_KEY')

Enter your ANTHROPIC_API_KEY ········


In [16]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

In [18]:
rewriter_prompt_template = """
Generate search query for the Chroma DB vector store
from a user question, allowing for a more accurate 
response through semantic search.
Just return the revised Chroma DB query, with quotes around it. 

User question: {user_question}
Revised Chroma DB query:
"""

rewriter_prompt = ChatPromptTemplate.from_template(
    rewriter_prompt_template) 

In [19]:
rewriter_chain = rewriter_prompt | llm | StrOutputParser()

### Retrieving content with the rewritten query

In [20]:
user_question ="Tell me some fun things I can do in Cornwall"

search_query = rewriter_chain.invoke(
    {"user_question": user_question})
print(search_query)

"fun activities and attractions in Cornwall"


In [21]:
improved_results = uk_granular_collection.similarity_search(
    query=search_query,k=3)
for doc in improved_results:
    print(doc)

page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 
 
 4.1   By train 
 
 
 
 
 
 
 4.2   By car 
 
 
 
 
 
 
 4.3   By plane 
 
 
 
 
 
 
 
 
 5   Get around 
 
 
 
 
 5.1   By bus 
 
 
 
 
 
 
 5.2   By train 
 
 
 
 
 
 
 
 
 6   See 
 
 
 
 
 6.1   National Trust properties 
 
 
 
 
 
 
 
 
 7   Do 
 
 
 
 
 7.1   Festivals 
 
 
 
 
 
 
 
 
 8   Drink 
 
 
 
 
 
 
 9   Stay safe 
 
 
 
 
 
 
 10   Go next 
 
 
 
 
 
 
 
 
 
 
 
 
 
 North Cornwall  is in  Cornwall . It includes much of the Cornish coast along the Celtic Sea and some top surfing areas.' metadata={'Header 2': 'Contents'}
page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 
 
 4.1   By train 
 
 
 
 
 
 
 4.2   By car 
 
 
 
 
 
 
 4.3   By bus 
 
 
 
 
 
 
 4.4   By plane 
 
 
 
 
 
 
 
 
 5   Get around 


### Combining everything in a single RAG chain

In [22]:
from langchain_core.runnables import RunnablePassthrough

In [23]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(
    rag_prompt_template) 

rewrite_retrieve_read_rag_chain = (
    {
        "context": {"user_question": RunnablePassthrough()} 
            | rewriter_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question

In [24]:
user_question = "Tell me some fun things I can do in Cornwall"

answer = rewrite_retrieve_read_rag_chain.invoke(user_question)
print(answer)

# Fun Things to Do in Cornwall

Based on the context provided, here are some enjoyable activities in Cornwall:

1. **Surfing** - North Cornwall includes some top surfing areas along the Celtic Sea coast

2. **Visit National Trust Properties** - Multiple regions mention National Trust properties and gardens as attractions

3. **Explore Cultural and Arts** - Cornwall is a popular destination for cultural tourism, with a long association with visual and written arts

4. **Visit the Eden Project** - Located near St. Austell in Mid-Cornwall, this features impressive biomes worth exploring

5. **Experience Archaeological Sites** - Cornwall has a wealth of archaeology to discover

6. **Attend Festivals** - Multiple regions list festivals as activities

7. **Enjoy the Scenery** - The stunning Cornish coastline along both the Celtic Sea and English Channel offers beautiful views and outdoor experiences

8. **Experience Cornish Heritage** - Explore the county's distinctive Celtic heritage, inclu

# Multiple query generation with MultiQueryRetriever

In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate

from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

## Implementing a custom MultiQueryRetriver

### Setting up the prompt

In [26]:
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task 
is to generate five different versions of the given 
user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user 
question, your goal is to help the user overcome some of 
the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template) 

### Setting up the multi-query parser

In [27]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  

questions_parser = LineListOutputParser()

### Setting up the chain to generate multiple queries

In [28]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

In [29]:
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

### Testing the Multi query gen chain

In [30]:
user_question = "Tell me some fun things I can do in Cornwall"

multiple_queries = multi_query_gen_chain.invoke(user_question)

In [31]:
multiple_queries

['# Alternative Question Versions',
 'What are the best recreational activities and attractions to visit in Cornwall?',
 'Where can I find entertainment and leisure activities in the Cornwall area?',
 'What outdoor adventures and tourist experiences are available in Cornwall?',
 'Can you recommend enjoyable places to visit and things to experience in Cornwall?',
 'What are popular and interesting ways to spend time in Cornwall for visitors and locals?']

### Setting up the MultiQueryRetriever

In [32]:
basic_retriever = uk_granular_collection.as_retriever()

multi_query_retriever = MultiQueryRetriever(
    retriever=basic_retriever, llm_chain=multi_query_gen_chain, 
    parser_key="lines" #A
)  
#A this is the key for the parsed output

### Using the multi_query retriever

In [33]:
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = multi_query_retriever.invoke(user_question)

In [34]:
retrieved_docs

[Document(id='98ebcf97-19ba-4f7c-bf01-2d6124bd7336', metadata={'Header 2': 'Contents'}, page_content='Contents \n move to sidebar \n hide \n \n \n \n \n Beginning \n \n \n \n \n \n 1 \n Understand \n \n \n \n \n Toggle Understand subsection \n \n \n \n \n \n 1.1 \n Flora and fauna \n \n \n \n \n \n \n \n \n 1.2 \n Climate \n \n \n \n \n \n \n \n \n 1.3 \n History \n \n \n \n \n \n \n \n \n \n \n 2 \n Get in \n \n \n \n \n Toggle Get in subsection \n \n \n \n \n \n 2.1 \n From London \n \n \n \n \n \n \n 2.1.1 \n By car \n \n \n \n \n \n \n \n \n 2.1.2 \n By train \n \n \n \n \n \n \n \n \n 2.1.3 \n By bus \n \n \n \n \n \n \n \n \n \n \n 2.2 \n From Kent/Medway Area \n \n \n \n \n \n \n 2.2.1 \n By car \n \n \n \n \n \n \n \n \n 2.2.2 \n By train \n \n \n \n \n \n \n \n \n 2.2.3 \n By bus \n \n \n \n \n \n \n \n \n \n \n 2.3 \n By air \n \n \n \n \n \n \n \n \n \n \n 3 \n Get around \n \n \n \n \n Toggle Get around subsection \n \n \n \n \n \n 3.1 \n Get a map \n \n \n \n \n \n \n \n \

## Using directly a standard MultiQueryRetriever 

In [35]:
std_multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=basic_retriever, llm=llm
)

In [36]:
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = std_multi_query_retriever.invoke(user_question)

In [37]:
retrieved_docs

[Document(id='98ebcf97-19ba-4f7c-bf01-2d6124bd7336', metadata={'Header 2': 'Contents'}, page_content='Contents \n move to sidebar \n hide \n \n \n \n \n Beginning \n \n \n \n \n \n 1 \n Understand \n \n \n \n \n Toggle Understand subsection \n \n \n \n \n \n 1.1 \n Flora and fauna \n \n \n \n \n \n \n \n \n 1.2 \n Climate \n \n \n \n \n \n \n \n \n 1.3 \n History \n \n \n \n \n \n \n \n \n \n \n 2 \n Get in \n \n \n \n \n Toggle Get in subsection \n \n \n \n \n \n 2.1 \n From London \n \n \n \n \n \n \n 2.1.1 \n By car \n \n \n \n \n \n \n \n \n 2.1.2 \n By train \n \n \n \n \n \n \n \n \n 2.1.3 \n By bus \n \n \n \n \n \n \n \n \n \n \n 2.2 \n From Kent/Medway Area \n \n \n \n \n \n \n 2.2.1 \n By car \n \n \n \n \n \n \n \n \n 2.2.2 \n By train \n \n \n \n \n \n \n \n \n 2.2.3 \n By bus \n \n \n \n \n \n \n \n \n \n \n 2.3 \n By air \n \n \n \n \n \n \n \n \n \n \n 3 \n Get around \n \n \n \n \n Toggle Get around subsection \n \n \n \n \n \n 3.1 \n Get a map \n \n \n \n \n \n \n \n \

# Step-back question

### Setting up the chain to generate the step-back question

In [38]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

In [39]:
step_back_prompt_template = """
Generate a less specific question (aka Step-back question) 
for the following detailed question, so that a wider context 
can be retrieved.
Detailed question: {detailed_question}
Step-back question:
"""

step_back_prompt = ChatPromptTemplate.from_template(
    step_back_prompt_template) 

In [40]:
step_back_question_gen_chain = step_back_prompt | llm | StrOutputParser()

### Testing the step-back-question generation chain

In [41]:
user_question = "Can you give me some tips for a trip to Brighton?"

step_back_question = step_back_question_gen_chain.invoke(user_question)

In [42]:
step_back_question

'# Step-back question:\n\n**What are some general travel tips for visiting UK coastal cities?**\n\n---\n\nThis step-back question is broader and would retrieve more general information about:\n- Visiting seaside destinations in the UK\n- Common travel planning considerations\n- Seasonal factors for coastal areas\n- General tourist tips\n\nThis wider context can then be narrowed down to Brighton-specific advice.'

### Incorporating step-back question generation chain into the RAG chain

In [43]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

step_back_question_rag_chain = (
    {
        "context": {"detailed_question": RunnablePassthrough()} 
           | step_back_question_gen_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

In [44]:
user_question = "Can you give me some tips for a trip to Brighton?"

answer = step_back_question_rag_chain.invoke(user_question)
print(answer)

# Tips for a Trip to Brighton

Based on the context provided, here are some helpful tips for visiting Brighton:

## Getting There
- Brighton is located 76 km (47 miles) south of London on the south-eastern coast of England
- You can arrive by train, car, bus, or plane

## Getting Around
- You have options including bikes, buses, trains, and taxis

## What to See & Do
- **Regency Architecture**: Brighton is known for its grand Regency-style buildings
- **The Pavilion**: A Grade-I Listed landmark with oriental-inspired architecture
- **Beaches**: Sussex has some of the cleanest beaches in the UK, with Brighton Beach being particularly popular
- **Brighton Festival & Fringe**: Features street performers, theatre groups, musicians, and guided walks
- **Museums**: Visit the Brighton Museum and Art Gallery or other town museums (some are free)

## Budget-Friendly Options
- Take advantage of 3,500 km of free walking paths and bridleways
- Enjoy the beaches for free
- Attend the Brighton Festi

# Hypotetical DocumentEmbeddings (HyDE)

### Setting up the chain to generate the hypotetical document associated to the user question

In [45]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

In [46]:
hyde_prompt_template = """
Write one sentence that could answer the provided question. 
Do not add anything else.
Question: {question}
Sentence:
"""

hyde_prompt = ChatPromptTemplate.from_template(hyde_prompt_template)

In [47]:
hyde_chain = hyde_prompt | llm | StrOutputParser()

### Testing the hyde generation chain

In [48]:
user_question = "What are the best beaches in Cornwall?"

hypotetical_document = hyde_chain.invoke(user_question)

In [49]:
hypotetical_document

'The best beaches in Cornwall include Perranporth, Fistral Beach, Polurrian Beach, and Watergate Bay, each offering excellent conditions for swimming, surfing, and coastal walks.'

### Incorporating hyde chain into the RAG chain

In [50]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
Only use the provided context to answer the question.
If you do not know the answer, just say I do not know. 

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

hyde_rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} 
           | hyde_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the hypotetical document
#B This is the original user question

In [51]:
user_question = "What are the best beaches in Cornwall?"

answer = hyde_rag_chain.invoke(user_question)
print(answer)

Based on the provided context, here are the best beaches in Newquay, North Cornwall:

**Popular Beaches:**
- **Fistral Beach** - Newquay's most popular beach, located west of Towards Head. Famous as a surf centre with lifeguards during summer months and hosts international surf competitions.
- **Towan Beach (Town Beach)** - Close to the town centre, accessible from the harbour with parking available and dogs allowed.
- **Great Western** - A popular family beach, accessible from Cliff Road.
- **Harbour** - Newquay's smallest beach, very popular with families.
- **Lusty Glaze Beach** - Offers a variety of shops and restaurants.
- **Crantock Beach** - A quiet beach located 2 km away from the city centre along the coastal path.
- **Watergate Bay** - Located north of Newquay.
- **Holywell Bay** - Located south of Crantock.

Newquay is noted as the "surf capital of Great Britain," so these beaches are particularly well-known for surfing. However, the context provided focuses specifically on 